# RAG Evaluation

> **If we cannot measure our RAG system, we don't really know whether it works.**

So far, we've built the major stages of a RAG system:

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embeddings
    ↓
Retrieval
    ↓
Hybrid Retrieval
    ↓
Reranking
    ↓
Context Engineering
    ↓
Grounded Generation
    ↓
Answer
```

We've also learned an important lesson:

> A RAG system can fail at different stages.

The retriever can miss the correct document.

The reranker can put the wrong chunk first.

Context construction can remove useful evidence.

The generation model can produce unsupported claims.

If we only inspect the final answer, we cannot reliably determine which component failed.

This notebook introduces **RAG evaluation**: measuring the different stages of a RAG system so that we can diagnose failures and improve the system systematically.

## What We'll Learn

By the end of this tutorial, you'll understand:

- Why RAG evaluation is different from ordinary application testing
- Evaluation datasets
- Ground-truth relevance
- Retrieval evaluation
- Recall@k
- Precision@k
- Mean Reciprocal Rank (MRR)
- Normalized Discounted Cumulative Gain (NDCG)
- Context evaluation
- Generation evaluation
- Faithfulness / groundedness
- Answer relevance
- End-to-end evaluation
- Error analysis
- Why evaluation should drive RAG optimization

The central idea is:

```text
RAG system
    ↓
Evaluation dataset
    ↓
Measure each stage
    ↓
Find failures
    ↓
Improve
    ↓
Measure again
```

# 1. Why RAG Evaluation Matters

A demo can work perfectly on five manually chosen questions and still fail badly in production.

For example:

```text
Question 1 → good
Question 2 → good
Question 3 → good
Question 4 → good
Question 5 → good
```

does not tell us how the system performs across:

- Different question types
- Ambiguous queries
- Long documents
- Exact identifiers
- Multi-hop questions
- Missing information
- Conflicting sources
- Different document structures

Evaluation gives us a repeatable way to answer:

> **How good is the system, and where is it failing?**

# 2. Evaluate the Pipeline, Not Just the Final Answer

A useful evaluation architecture is:

```text
                     RAG System
                         │
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
      Retrieval       Context       Generation
      Evaluation      Evaluation    Evaluation
          │              │              │
          └──────────────┼──────────────┘
                         ↓
                  End-to-End Quality
```

This separation matters.

If the final answer is wrong, we can ask:

```text
Was the evidence missing?
        ↓
Retrieval problem

Was the evidence available but badly selected?
        ↓
Context problem

Was the evidence correct but the answer unsupported?
        ↓
Generation problem
```

Without stage-level evaluation, these failures are easy to confuse.

# 3. Evaluation Dataset

Before we can measure a RAG system, we need questions to test it.

A basic evaluation record might contain:

```python
{
    "question": "...",
    "relevant_document_ids": [...],
    "reference_answer": "..."
}
```

The fields depend on the metric we want to calculate.

For retrieval evaluation, we need to know which documents are relevant.

For answer evaluation, we may also need a reference answer or another reliable evaluation method.

In [1]:
evaluation_dataset = [
    {
        "question": "How long do customers have to request a refund?",
        "relevant_document_ids": ["refund-policy"],
        "reference_answer": "Customers can request a refund within 30 days of purchase.",
    },
    {
        "question": "How long are approved refunds normally processed?",
        "relevant_document_ids": ["refund-process"],
        "reference_answer": "Approved refunds are normally processed within 7 business days.",
    },
    {
        "question": "Which payment method receives the refund?",
        "relevant_document_ids": ["refund-method"],
        "reference_answer": "The refund is returned to the original payment method.",
    },
]

This dataset is intentionally small.

In real systems, an evaluation set should represent the queries the application is expected to handle.

A good dataset can contain:

- Common questions
- Difficult questions
- Edge cases
- Queries with exact terminology
- Queries requiring multiple pieces of evidence
- Questions with no answer in the corpus
- Questions involving conflicting or outdated information

# 4. Ground Truth

Evaluation requires some notion of a trusted answer.

For retrieval, the ground truth might be:

```text
Question
    ↓
Relevant document(s)
```

For generation, it might be:

```text
Question
    ↓
Reference answer
```

But ground truth does not have to mean that there is always exactly one correct chunk or one exact string.

Some questions can have:

```text
Multiple relevant documents
Multiple acceptable answers
Multiple valid phrasings
```

Therefore, evaluation data should represent the actual information needs of the application.

# 5. Retrieval Evaluation

Let's start with the retrieval stage.

Suppose the relevant document is:

```text
refund-policy
```

and our retriever returns:

```text
1. shipping
2. refund-process
3. refund-policy
4. support-hours
5. refund-method
```

The correct document appears at rank 3.

That gives us a way to measure retrieval quality.

# 6. Recall@k

**Recall@k** asks:

> Did we retrieve at least one relevant document in the top `k` results?

For one query:

```text
Relevant document: refund-policy

Top 3:
1. shipping
2. refund-process
3. refund-policy
```

The answer is:

```text
Recall@3 = 1
```

because the relevant document appears within the top three.

If it were outside the top three:

```text
Recall@3 = 0
```

Across multiple questions, Recall@k is the proportion of questions for which the relevant evidence was retrieved within the top k.

In [2]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved_top_k = set(retrieved_ids[:k])
    relevant_ids = set(relevant_ids)

    return int(bool(retrieved_top_k & relevant_ids))


retrieved = [
    "shipping",
    "refund-process",
    "refund-policy",
    "support-hours",
]

print(recall_at_k(
    retrieved,
    ["refund-policy"],
    k=3
))

1


Recall@k is particularly useful for the first-stage retriever because it answers:

> **Did the evidence make it into the candidate set?**

If Recall@20 is poor, reranking cannot fix the problem.

This is why we emphasized earlier:

> Reranking cannot recover documents that retrieval never found.

# 7. Precision@k

Recall asks:

> Did we find relevant information?

Precision asks:

> How much of what we retrieved is relevant?

Suppose:

```text
Top 5 retrieved:
1. relevant ✓
2. relevant ✓
3. irrelevant ✗
4. irrelevant ✗
5. irrelevant ✗
```

Then:

```text
Precision@5 = 2 / 5 = 0.4
```

Precision is useful when we care about the amount of irrelevant material entering the candidate set or final context.

In [3]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_ids = set(relevant_ids)

    if k == 0:
        return 0.0

    relevant_count = sum(
        item in relevant_ids
        for item in retrieved_top_k
    )

    return relevant_count / k


print(precision_at_k(
    ["refund-policy", "shipping", "support-hours"],
    ["refund-policy"],
    k=3
))

0.3333333333333333


Recall and precision often pull in different directions.

```text
Increase k
   ↓
Recall may increase
   ↓
Precision may decrease
```

This is one reason we use multiple stages in RAG.

The first stage can prioritize recall.

Later stages can improve precision.

# 8. Mean Reciprocal Rank (MRR)

Sometimes we care strongly about **where the first relevant result appears**.

MRR captures this.

For one query:

```text
1. irrelevant
2. irrelevant
3. relevant
```

The reciprocal rank is:

```text
1 / 3
```

If the relevant result is first:

```text
1 / 1 = 1
```

If the relevant result is fifth:

```text
1 / 5 = 0.2
```

Across multiple queries, MRR is the average reciprocal rank of the first relevant result.

In [4]:
def reciprocal_rank(retrieved_ids, relevant_ids):
    relevant_ids = set(relevant_ids)

    for rank, item in enumerate(retrieved_ids, start=1):
        if item in relevant_ids:
            return 1.0 / rank

    return 0.0


print(reciprocal_rank(
    ["shipping", "support-hours", "refund-policy"],
    ["refund-policy"]
))

0.3333333333333333


MRR is especially intuitive for question-answering systems where we care about getting a useful source near the top of the ranking.

# 9. NDCG

**Normalized Discounted Cumulative Gain (NDCG)** is useful when:

- Multiple documents can be relevant
- Relevance can have different degrees
- Ranking position matters

Instead of simply saying:

```text
relevant / irrelevant
```

we can assign graded relevance:

```text
3 → highly relevant
2 → relevant
1 → somewhat relevant
0 → irrelevant
```

NDCG gives more credit to highly relevant documents appearing near the top.

We won't implement a complete production NDCG evaluator here yet.

The important concept is:

> **Ranking quality is not only about whether relevant documents appear; their positions matter.**

# 10. Retrieval Metrics Summary

We now have several complementary metrics:

| Metric | Main question |
|---|---|
| Recall@k | Did relevant evidence appear in the top k? |
| Precision@k | How much of the top k was relevant? |
| MRR | How high was the first relevant result? |
| NDCG | How good was the overall ranked ordering? |

No single metric describes every aspect of retrieval quality.

# 11. Evaluate the Retriever on Multiple Questions

We can calculate Recall@k across our evaluation dataset.

For demonstration, imagine these are the retrieved rankings produced by a retriever.

In [5]:
retrieval_results = [
    ["refund-policy", "shipping", "support-hours"],
    ["shipping", "refund-process", "refund-policy"],
    ["refund-method", "refund-policy", "shipping"],
]

for example, expected in zip(
    retrieval_results,
    evaluation_dataset
):
    score = recall_at_k(
        example,
        expected["relevant_document_ids"],
        k=3,
    )

    print(
        expected["question"],
        "→ Recall@3:",
        score
    )

How long do customers have to request a refund? → Recall@3: 1
How long are approved refunds normally processed? → Recall@3: 1
Which payment method receives the refund? → Recall@3: 1


For a real experiment, we would run the actual retriever over every evaluation question and aggregate the results.

For example:

```text
Recall@3 = 0.67
Recall@5 = 0.83
Recall@10 = 0.97
```

That tells us how candidate count affects retrieval coverage.

# 12. Context Evaluation

Retrieval evaluation asks:

> Did we retrieve the relevant evidence?

But the generation model does not receive the entire retrieval result.

Context evaluation asks:

> **Did the final context contain the evidence needed to answer the question?**

For example:

```text
Retriever
    ↓
10 candidates
    ↓
Context selection
    ↓
3 chunks
```

If the relevant chunk was retrieved but removed during context construction:

```text
Retrieval ✓
Context ✗
```

This is a different failure.

# 13. Context Recall

One useful concept is **context recall**:

> Did the final context contain the information required to answer the question?

For a simple evaluation dataset, we can approximate this by checking whether the known relevant document appears in the final context.

In [6]:
def context_contains_relevant(
    context_document_ids,
    relevant_document_ids
):
    return int(
        bool(
            set(context_document_ids)
            & set(relevant_document_ids)
        )
    )


print(context_contains_relevant(
    ["refund-policy", "refund-process"],
    ["refund-policy"]
))

1


In a production evaluation framework, context relevance can be much more nuanced.

We may ask:

- Does the context contain the needed evidence?
- Is the evidence sufficient?
- Is there too much irrelevant information?
- Are sources duplicated?
- Are conflicting sources present?
- Is the evidence in a usable form?

# 14. Generation Evaluation

Now we reach the final stage.

Suppose the system has:

```text
Correct context
    ↓
Generated answer
```

We need to determine whether the answer actually uses the evidence correctly.

Important dimensions include:

### Faithfulness / Groundedness

Are the answer's claims supported by the provided context?

### Answer relevance

Does the answer actually address the user's question?

### Correctness

Does the answer match a trusted reference answer or otherwise satisfy the task?

These dimensions are related but not identical.

# 15. Faithfulness

Consider:

Context:

```text
Customers can request a refund within 30 days of purchase.
```

Answer:

```text
Customers can request a refund within 30 days of purchase.
```

The answer is supported.

Now:

```text
Customers can request a refund within 30 days,
and all refund requests are automatically approved.
```

The second claim is not supported.

A faithfulness evaluation therefore asks:

> **Are the claims in the answer entailed by the available evidence?**

This can be evaluated with human judgments, model-based evaluators, or more specialized evaluation systems.

# 16. Answer Relevance

An answer can be grounded but still fail to answer the question.

Question:

```text
How long do customers have to request a refund?
```

Context:

```text
Customers can request a refund within 30 days.
```

Answer:

```text
Refunds are returned to the original payment method.
```

The answer may be true according to some available context, but it does not answer the question.

So:

```text
Faithfulness
    ≠
Answer relevance
```

We need to measure both when appropriate.

# 17. Reference Answers

For some tasks, we can compare the generated answer against a reference answer.

For example:

```text
Reference:
Customers can request a refund within 30 days.

Generated:
Customers have 30 days from purchase to request a refund.
```

These strings are not identical, but the generated answer is semantically equivalent.

This is why exact string matching is often insufficient for natural-language evaluation.

Depending on the task, evaluation can use:

- Human judgment
- Semantic similarity
- LLM-based evaluation
- Rule-based checks
- Domain-specific validators

The evaluation method should match the risk and requirements of the application.

# 18. End-to-End Evaluation

Ultimately, we care about the user's experience.

A useful end-to-end evaluation record might look like:

```text
Question
    ↓
Retrieved documents
    ↓
Reranked documents
    ↓
Final context
    ↓
Generated answer
    ↓
Sources
    ↓
Evaluation results
```

This allows us to trace a bad answer back to its cause.

# 19. Error Analysis

Metrics tell us **that** something is wrong.

Error analysis helps us understand **why**.

Suppose Recall@5 is low.

Possible causes:

```text
Wrong embedding model
        ↓
Poor chunking
        ↓
Query mismatch
        ↓
Insufficient candidate count
        ↓
Metadata filtering
```

Suppose retrieval is strong but answers are poor:

```text
Retrieval ✓
Context ✗
```

or:

```text
Context ✓
Generation ✗
```

This is why a production evaluation system should preserve intermediate results.

# 20. Build an Evaluation Record

Let's create a structure that keeps the pipeline traceable.

In [7]:
evaluation_record = {
    "question": "How long do customers have to request a refund?",

    "retrieved_documents": [
        "refund-policy",
        "shipping",
        "support-hours",
    ],

    "reranked_documents": [
        "refund-policy",
        "shipping",
        "support-hours",
    ],

    "context_documents": [
        "refund-policy",
    ],

    "generated_answer": (
        "Customers can request a refund within "
        "30 days of purchase."
    ),

    "reference_answer": (
        "Customers can request a refund within "
        "30 days of purchase."
    ),
}

evaluation_record

{'question': 'How long do customers have to request a refund?',
 'retrieved_documents': ['refund-policy', 'shipping', 'support-hours'],
 'reranked_documents': ['refund-policy', 'shipping', 'support-hours'],
 'context_documents': ['refund-policy'],
 'generated_answer': 'Customers can request a refund within 30 days of purchase.',
 'reference_answer': 'Customers can request a refund within 30 days of purchase.'}

This kind of trace becomes extremely useful when debugging.

Instead of storing only:

```text
Question → Answer
```

we store:

```text
Question
→ Retrieval
→ Reranking
→ Context
→ Generation
→ Evaluation
```

That gives us an evidence trail for the system itself.

# 21. Evaluation-Driven Optimization

Now we can establish a disciplined optimization loop:

```text
Baseline RAG
     ↓
Evaluate
     ↓
Find bottleneck
     ↓
Change one component
     ↓
Evaluate again
     ↓
Compare
```

For example:

```text
Baseline
Recall@5 = 0.72

Change chunking strategy

Recall@5 = 0.84
```

We have evidence that the change helped retrieval.

Or:

```text
Baseline
Faithfulness = 0.81

Add stronger grounding policy

Faithfulness = 0.88
```

Again, we can measure the effect.

# 22. Never Optimize Against One Metric

Suppose a change improves:

```text
Recall@10
```

but increases:

```text
Latency
```

or decreases:

```text
Precision@5
```

The change may not actually be an improvement for the product.

Production RAG is a multi-objective optimization problem involving:

- Retrieval quality
- Context quality
- Answer quality
- Latency
- Cost
- Reliability
- Freshness
- Security

The correct trade-offs depend on the application.

# 23. Evaluation Dataset Design

A weak evaluation set can produce misleading confidence.

A stronger dataset should cover the application's real distribution.

Consider including:

### Easy
Questions answered directly by one chunk.

### Difficult
Questions where relevant information is harder to retrieve.

### Exact-match
Queries containing names, IDs, codes, or technical terms.

### Multi-document
Questions requiring information from multiple sources.

### Unanswerable
Questions where the corpus does not contain the answer.

### Conflicting
Questions involving different versions or sources.

### Long-context
Questions where relevant information is buried among irrelevant content.

This gives us a much more realistic picture of system behavior.

# 24. Evaluation in Development vs. Production

Offline evaluation is not the end.

A production system changes over time:

```text
New documents
New users
New queries
New models
New prompts
New indexes
```

Therefore, evaluation should be continuous.

A useful workflow is:

```text
Offline evaluation
        ↓
Release candidate
        ↓
Production monitoring
        ↓
Collect failures
        ↓
Add representative cases
        ↓
Update evaluation set
        ↓
Re-evaluate
```

Your evaluation dataset should evolve with the system.

# 25. The RAG Evaluation Stack

We can now think of evaluation as several layers:

```text
                    RAG Evaluation
                          │
        ┌─────────────────┼─────────────────┐
        ↓                 ↓                 ↓
     Retrieval         Context          Generation
     Quality           Quality            Quality
        │                 │                 │
   Recall@k          Context recall     Faithfulness
   Precision@k       Context relevance  Answer relevance
   MRR                                   Correctness
   NDCG
        └─────────────────┼─────────────────┘
                          ↓
                   End-to-End Quality
                          ↓
                    Error Analysis
                          ↓
                     Optimization
```

This is the foundation for serious RAG engineering.

# 26. What We Have Achieved

Our RAG pipeline now has measurable stages:

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embeddings
    ↓
Retrieval
    ↓
Hybrid Retrieval
    ↓
Reranking
    ↓
Context Engineering
    ↓
Grounded Generation
    ↓
Evaluation
```

And evaluation gives us the ability to ask:

```text
Did retrieval find the evidence?
Did reranking order it correctly?
Did context preserve it?
Did generation use it?
Did the final answer satisfy the question?
```

That is a major step beyond building a RAG demo.

# Key Takeaways

1. A RAG system should be evaluated as a pipeline, not just by inspecting final answers.
2. Evaluation requires a representative dataset.
3. Retrieval evaluation measures whether relevant evidence is found.
4. Recall@k measures whether relevant evidence appears within the top k.
5. Precision@k measures how much of the retrieved set is relevant.
6. MRR measures the rank of the first relevant result.
7. NDCG evaluates ranked results when relevance can be graded.
8. Context evaluation asks whether the evidence actually reaches the LLM.
9. Generation evaluation includes faithfulness, relevance, and correctness.
10. A grounded answer can still be irrelevant to the question.
11. Error analysis connects metrics to actual engineering problems.
12. Evaluation should drive optimization rather than intuition.
13. Improving one metric can hurt latency, cost, or another quality metric.
14. Production evaluation should evolve as users, documents, and systems change.
15. Intermediate traces make RAG failures much easier to diagnose.

The mental model is:

```text
Measure
   ↓
Diagnose
   ↓
Change
   ↓
Measure again

Don't guess.
Evaluate.
```

# What's Next?

We now understand the core RAG pipeline and how to evaluate it.

Before moving into more advanced retrieval strategies, we need to understand one of the most important parts of a RAG system:

**How do we get information from real-world documents into our RAG pipeline?**

The next stage is **Document Ingestion**.

We'll investigate how to:

* Load documents into our system
* Parse PDFs and other document formats
* Extract text and structure
* Handle OCR when documents are scanned
* Clean extracted content
* Preserve useful document structure
* Attach metadata and provenance
* Prepare documents for downstream chunking and retrieval

The key question changes from:

> **How do we build a RAG pipeline?**

to:

> **How do we reliably turn real-world documents into usable knowledge for a RAG system?**

That is the beginning of **document ingestion engineering**.
